# Governance Dashboard — QuickSight Analysis for Drift Monitoring

This notebook creates or refreshes the Lab 5E QuickSight dashboard. The implementation lives in `src/governance/create_governance_dashboard.py`; the notebook stays a short driver.

The feature sheet compares normalized **drift magnitude**. A magnitude of `1.0` is the feature's method threshold, and values strictly above `1.0` are drifted. The detail table retains raw score, method, threshold, and direction for audit. Dataset-level `drifted_columns_share` remains the fraction of features that drifted.

Inference rows and drift runs join by `monitoring_run_id`, which Lab 5D stamps at scoring time. The join does not infer run membership from overlapping timestamps.

## What the build creates

The module creates one Athena data source, five direct-query datasets, a verified `feature_drift_detail` Athena view, a three-sheet analysis, and a published dashboard. It reconciles every discovered QuickSight user onto new and existing resources without revoking unrelated principals.

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

import boto3

project_root = Path.cwd()
_env_dir = next((p for p in [project_root, *project_root.parents]
                 if (p / '.env').exists()), None)
if _env_dir is None:
    raise RuntimeError('No .env found. Run lab0-setup/setup.ipynb first.')
load_dotenv(_env_dir / '.env', override=True)

_lab5_str = str(project_root)
if _lab5_str not in sys.path:
    sys.path.insert(0, _lab5_str)
for _k in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
    del sys.modules[_k]

from src.config.config import AWS_DEFAULT_REGION, QUICKSIGHT_IDENTITY_REGION
import src.governance.create_governance_dashboard as gov

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'bank-marketing-prediction')
ACCOUNT_ID = os.environ.get('ACCOUNT_ID') or \
    boto3.client('sts', region_name=AWS_DEFAULT_REGION).get_caller_identity()['Account']

print(f'✓ Loaded {_env_dir / ".env"}')
print(f'  PROJECT_NAME               = {PROJECT_NAME}')
print(f'  ACCOUNT_ID                 = {ACCOUNT_ID}')
print(f'  AWS_DEFAULT_REGION         = {AWS_DEFAULT_REGION}')
print(f'  QUICKSIGHT_IDENTITY_REGION = {QUICKSIGHT_IDENTITY_REGION}')

## 2. QuickSight prerequisite and automatic permission preflight

QuickSight **Enterprise** must be subscribed once per account. Subscription is billable and remains an explicit console or deployment choice.

The IAM stack always creates two project-scoped managed policies: `<ProjectName>-QuickSightS3DataLakeAccess` and `<ProjectName>-QuickSightAthenaAccess`. If subscription existed before deployment, CloudFormation attaches them normally. If subscription happened after infrastructure provisioning, the dashboard build resolves only this account and project's policy ARNs, attaches only missing policies to `service-role/aws-quicksight-service-role-v0`, verifies both attachments, and stops before creating assets if anything is wrong. No manual repair or broad S3 policy is needed.

For automated provisioning, `SUBSCRIBE_QUICKSIGHT=true QUICKSIGHT_NOTIFICATION_EMAIL=you@example.com ./scripts/deploy-workshop.sh` subscribes before the IAM stack. The flag defaults to `false` so deployment never creates a billable subscription silently.

In [ ]:
quicksight_admin = boto3.client('quicksight', region_name=QUICKSIGHT_IDENTITY_REGION)
subscription = gov.check_quicksight_subscription(
    account_id=ACCOUNT_ID, quicksight_admin_client=quicksight_admin,
)

if subscription['subscribed']:
    print(f"✓ QuickSight active (Edition: {subscription['edition']})")
    if subscription['edition'] == 'STANDARD':
        print('  ⚠ Upgrade to Enterprise before building the dashboard.')
elif subscription['reason'] == 'access-denied':
    print('✗ This role cannot check QuickSight. Apply the Lab 5E IAM statement')
    print('  from templates/2-iam.yaml, then rerun this cell.')
else:
    print('✗ Subscribe this account to QuickSight Enterprise, then rerun this cell.')

## 3. Build or refresh the dashboard

In [ ]:
from botocore.exceptions import ClientError

result = None
try:
    result = gov.create_dashboard(region=AWS_DEFAULT_REGION)
except gov.QuickSightUnavailable as exc:
    print(f'✗ Dashboard not created — {exc}')
except ClientError as exc:
    if exc.response['Error']['Code'] not in ('AccessDeniedException', 'UnrecognizedClientException'):
        raise
    print(f'✗ QuickSight denied {exc.operation_name}: {exc.response["Error"]["Message"]}')
    print('  Apply the full Lab 5E statement from templates/2-iam.yaml.')

if result:
    attached = result['quicksight_service_role_permissions']['attached_now']
    print('✓ QuickSight service-role preflight passed')
    print(f'  Policies attached by this run: {len(attached)}')
    print(f"  Dashboard URL: {result['dashboard_url']}")
    print(f"  Analysis ARN:  {result['analysis_arn']}")

## Optional cleanup

In [ ]:
CONFIRM_DELETE = False  # set True to delete all governance QuickSight resources

if CONFIRM_DELETE:
    outcome = gov.delete_dashboard(region=AWS_DEFAULT_REGION)
    print('Deleted:  ', outcome['deleted'])
    print('Not found:', outcome['not_found'])
    print('Errors:   ', outcome['errors'])
else:
    print('Cleanup skipped — set CONFIRM_DELETE = True to run.')